# CosQA E5 + cross-encoder reranking

This notebook evaluates `intfloat/e5-base-v2` as the first-stage retriever and `cross-encoder/ms-marco-MiniLM-L6-v2` as a second-stage scorer on the pinned `CoIR-Retrieval/cosqa` data. It evaluates both the unchanged E5 ranking and the reranked candidate pool at `nDCG@10`.

The default execution mode is a small **real-model smoke run**. Smoke output proves wiring only and is not benchmark evidence. Set `E5_RERANK_MODE=benchmark` for the complete declared test-qrels query population and corpus.

In [ ]:
# Install code-retrieval/requirements.txt once before running this notebook if needed.
# In Colab, uncomment the next line and restart the runtime if pip updates torch/numpy.
# %pip install -r ../requirements.txt

from pathlib import Path
import hashlib
import json
import os
import sys
import time

# This experiment is PyTorch-only; avoid importing TensorFlow/Keras through Transformers.
os.environ.setdefault('USE_TF', '0')

workspace_root = Path.cwd()
while workspace_root != workspace_root.parent and not (workspace_root / 'code-retrieval' / 'src').exists():
    workspace_root = workspace_root.parent
project_dir = workspace_root / 'code-retrieval'
if not (project_dir / 'src').exists():
    project_dir = Path.cwd()
    workspace_root = project_dir.parent

os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))
sys.path.insert(0, str(project_dir / 'scripts'))
from run_benchmark_notebook import require_pinned_runtime

from e5_baseline import (
    E5Encoder,
    load_cosqa,
    package_versions,
    select_run_data,
)
from e5_rerank import (
    CrossEncoderReranker,
    RerankConfig,
    build_comparison_result,
    candidate_pool_preserved,
    environment_metadata,
    evaluate_ndcg_at_10,
    expected_rerank_cache_metadata,
    load_valid_embedding_cache,
    load_valid_json_cache,
    rank_with_faiss,
    rerank_cache_paths,
    rerank_rankings,
    rerank_run_identity,
    save_embedding_cache,
    save_json_cache,
    score_candidate_pool,
    set_seed,
    write_result_artifacts,
)
from e5_hyde_rerank import reciprocal_rank_fusion

print('Project directory:', project_dir.resolve())
print('Python:', sys.version.split()[0])

## 1. Environment and explicit controls

The first-stage controls match AZH-514: pinned CosQA data, E5-base-v2, text-only query/passage inputs, the paper's Faiss `IndexFlat` exact-retrieval path, candidate depth `1000`, and COIR `nDCG@10`. The second-stage model is pinned to the observed Hugging Face revision `233902d25c440f23af6f7d6e94d2946bac0bee0a`.

In [ ]:
RUN_MODE = os.environ.get('E5_RERANK_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'benchmark'}:
    raise ValueError('E5_RERANK_MODE must be smoke or benchmark')
if RUN_MODE == 'benchmark':
    require_pinned_runtime()

batch_size = int(os.environ.get('E5_RERANK_E5_BATCH_SIZE', '32' if RUN_MODE == 'smoke' else '128'))
reranker_batch_size = int(os.environ.get('E5_RERANK_BATCH_SIZE', '32' if RUN_MODE == 'smoke' else '128'))
torch_threads = int(os.environ.get('E5_RERANK_TORCH_THREADS', '8'))
if torch_threads > 0:
    import torch
    torch.set_num_threads(torch_threads)

qrels_split = os.environ.get('E5_RERANK_QRELS_SPLIT', 'test')
fusion_weights = (0.8, 0.2)
if RUN_MODE == 'benchmark' and qrels_split == 'test':
    selection_path = Path('artifacts/e5_hyde_rerank/validation/weight_selection.json')
    if not selection_path.is_file():
        raise RuntimeError('Run valid-split fusion selection before evaluating test qrels')
    selection = json.loads(selection_path.read_text(encoding='utf-8'))
    if selection.get('selection_qrels_split') != 'valid' or selection.get('query_count') != 500 or selection.get('candidate_depth') != 1000:
        raise RuntimeError('Test fusion weights must come from the complete valid split')
    fusion_weights = tuple(selection['systems']['e5_rerank']['selected']['weights'])

config = RerankConfig(
    run_mode=RUN_MODE,
    batch_size=batch_size,
    reranker_batch_size=reranker_batch_size,
    cache_dir='artifacts/e5_rerank/cache',
    artifact_dir='artifacts/e5_rerank',
    qrels_split=qrels_split,
    fusion_weights=fusion_weights,
)
if config.candidate_depth != 1000:
    raise RuntimeError('Reranking must use the benchmark candidate depth of 1000')
set_seed(config.seed)

print(json.dumps(config.as_dict(), indent=2, sort_keys=True))
print('Installed packages:')
print(json.dumps(package_versions(['coir-eval', 'datasets', 'faiss-cpu', 'numpy', 'pytrec-eval-terrier', 'sentence-transformers', 'torch', 'transformers']), indent=2, sort_keys=True))
print('Hardware/runtime:')
environment = environment_metadata(config, repo_root=workspace_root)
print(json.dumps(environment, indent=2, sort_keys=True, default=str))

## 2. Load and inspect the real CosQA schema

The loader reads the separate `corpus`, `queries`, and `default/test` configurations from one pinned revision. Titles are excluded from model inputs to preserve the paper-compatible COIR text-only path.

In [ ]:
data = load_cosqa(config)
run_data = select_run_data(data, config)
if config.run_mode == 'benchmark' and (len(run_data.queries) != len(data.queries) or len(run_data.corpus) != len(data.corpus)):
    raise RuntimeError('Benchmark mode must use complete CosQA test queries and corpus')
if config.run_mode == 'benchmark' and len(run_data.qrels) != len(data.qrels):
    raise RuntimeError('Benchmark mode must retain every CosQA test-qrels query')

print(json.dumps(data.schema, indent=2, sort_keys=True, default=str))
print(json.dumps({
    'run_mode': config.run_mode,
    'corpus_count': len(run_data.corpus),
    'query_count': len(run_data.queries),
    'qrels_query_count': len(run_data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in run_data.qrels.values()),
    'exclusions': run_data.exclusions,
}, indent=2))
print('Corpus example:', next(iter(run_data.corpus.items())))
print('Query example:', next(iter(run_data.queries.items())))
print('Qrels example:', next(iter(run_data.qrels.items())))

## 3. Run identity and cache paths

All first- and second-stage caches are isolated under an identity that includes the complete configuration, reranker revision, code version, repository commit, and ordered source-ID fingerprints.

In [ ]:
notebook_path = project_dir / 'notebooks' / 'e5_rerank_experiment.ipynb'
notebook_sha256 = hashlib.sha256(notebook_path.read_bytes()).hexdigest() if notebook_path.exists() else None
identity = rerank_run_identity(config, repo_root=workspace_root, notebook_sha256=notebook_sha256, environment=environment)
paths = rerank_cache_paths(config, identity)
print('Run identity:', identity)
print('Notebook SHA-256:', notebook_sha256)
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))

## 4. Encode corpus and queries with E5

E5 receives `passage: ` for corpus text and `query: ` for query text. No title, answer, label, or target-document information is passed to either retrieval stage.

In [ ]:
encoder = E5Encoder(config)
corpus_ids = list(run_data.corpus)
corpus_texts = [run_data.corpus[doc_id]['text'] for doc_id in corpus_ids]
corpus_metadata = expected_rerank_cache_metadata(
    config, identity=identity, kind='corpus_embeddings', ids=corpus_ids, repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
    environment=environment
)
corpus_embeddings = load_valid_embedding_cache(paths['corpus_embeddings'], paths['corpus_metadata'], corpus_metadata)
if corpus_embeddings is None:
    corpus_started = time.perf_counter()
    corpus_embeddings = encoder.encode_corpus(corpus_texts)
    corpus_seconds = time.perf_counter() - corpus_started
    save_embedding_cache(paths['corpus_embeddings'], paths['corpus_metadata'], corpus_embeddings, corpus_metadata)
else:
    corpus_seconds = 0.0
print('Corpus embeddings:', corpus_embeddings.shape, 'seconds:', round(corpus_seconds, 3))

query_ids = list(run_data.queries)
query_texts = [run_data.queries[query_id] for query_id in query_ids]
query_metadata = expected_rerank_cache_metadata(
    config, identity=identity, kind='query_embeddings', ids=query_ids, repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
    environment=environment
)
query_embeddings = load_valid_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_metadata)
if query_embeddings is None:
    query_started = time.perf_counter()
    query_embeddings = encoder.encode_queries(query_texts)
    query_seconds = time.perf_counter() - query_started
    save_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_embeddings, query_metadata)
else:
    query_seconds = 0.0
print('Query embeddings:', query_embeddings.shape, 'seconds:', round(query_seconds, 3))

## 5. Exact E5 first-stage retrieval

Paper-faithful Faiss `IndexFlat` search creates the 1000-document candidate pool used by both systems. The reranker cannot add documents beyond this pool.

In [ ]:
ranking_ids = query_ids + ['__corpus__'] + corpus_ids
ranking_metadata = expected_rerank_cache_metadata(
    config, identity=identity, kind='rankings', ids=ranking_ids, repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
    environment=environment
)
rankings = load_valid_json_cache(paths['rankings'], paths['rankings_metadata'], ranking_metadata)
if rankings is None:
    ranking_started = time.perf_counter()
    rankings = rank_with_faiss(
        query_embeddings,
        corpus_embeddings,
        query_ids,
        corpus_ids,
        top_k=config.candidate_depth,
    )
    ranking_seconds = time.perf_counter() - ranking_started
    save_json_cache(paths['rankings'], paths['rankings_metadata'], rankings, ranking_metadata)
else:
    ranking_seconds = 0.0
print('First-stage ranking queries:', len(rankings))
print('First-stage example:', next(iter(rankings.items())))

## 6. Score and reorder only the E5 candidate pool

The cross-encoder sees raw query/corpus text pairs for the E5 candidates. Its score order is deterministic on ties by corpus ID, and the candidate IDs are checked before evaluation.

In [ ]:
rerank_ids = []
for query_id, candidate_scores in rankings.items():
    rerank_ids.append(query_id)
    rerank_ids.extend(candidate_scores)
rerank_metadata = expected_rerank_cache_metadata(
    config, identity=identity, kind='reranked_rankings', ids=rerank_ids, repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
    environment=environment
)
reranked_rankings = load_valid_json_cache(
    paths['reranked_rankings'], paths['reranked_rankings_metadata'], rerank_metadata
)
if reranked_rankings is None:
    reranker = CrossEncoderReranker(config)
    rerank_started = time.perf_counter()
    reranker_scores = score_candidate_pool(
        run_data.queries, run_data.corpus, rankings, reranker.score_pairs
    )
    reranked_rankings = rerank_rankings(rankings, reranker_scores)
    reranking_seconds = time.perf_counter() - rerank_started
    save_json_cache(
        paths['reranked_rankings'], paths['reranked_rankings_metadata'], reranked_rankings, rerank_metadata
    )
else:
    reranking_seconds = 0.0

# Re-validate cached scores and canonicalize their deterministic order.
reranked_rankings = rerank_rankings(rankings, reranked_rankings)
pool_preserved = candidate_pool_preserved(rankings, reranked_rankings)
if not pool_preserved:
    raise RuntimeError('Reranked results changed the E5 candidate pool')
print('Candidate pool preserved:', pool_preserved)
print('Reranked example:', next(iter(reranked_rankings.items())))

fused_metadata = expected_rerank_cache_metadata(
    config, identity=identity, kind='fused_rankings', ids=rerank_ids, repo_root=workspace_root,
    notebook_sha256=notebook_sha256, environment=environment,
)
fused_rankings = load_valid_json_cache(
    paths['fused_rankings'], paths['fused_rankings_metadata'], fused_metadata
)
if fused_rankings is None:
    fusion_started = time.perf_counter()
    fused_rankings = reciprocal_rank_fusion(
        [rankings, reranked_rankings],
        weights=config.fusion_weights,
        top_k=config.candidate_depth,
        rrf_k=config.rrf_k,
    )
    fusion_seconds = time.perf_counter() - fusion_started
    save_json_cache(
        paths['fused_rankings'], paths['fused_rankings_metadata'], fused_rankings, fused_metadata
    )
else:
    fusion_seconds = 0.0

pool_preserved = candidate_pool_preserved(rankings, fused_rankings)
if not pool_preserved:
    raise RuntimeError('Fused results changed the E5 candidate pool')
print('E5 candidate pool preserved by fused ranking:', pool_preserved)
print('First fused ranking:', next(iter(fused_rankings.items())))

## 7. Evaluate the E5-anchored ranking with official COIR

Evaluate the E5 baseline, raw cross-encoder ranking, and validation-selected
E5/reranker RRF ranking with the same qrels, evaluator, and `nDCG@10` cutoff.
The comparison score is the fused ranking; the raw reranker score is a diagnostic.

In [ ]:
evaluation_started = time.perf_counter()
baseline_metric = evaluate_ndcg_at_10(run_data.qrels, rankings, cutoff=10)
baseline_evaluation_seconds = time.perf_counter() - evaluation_started

evaluation_started = time.perf_counter()
raw_reranked_metric = evaluate_ndcg_at_10(run_data.qrels, reranked_rankings, cutoff=10)
raw_reranked_evaluation_seconds = time.perf_counter() - evaluation_started

evaluation_started = time.perf_counter()
reranked_metric = evaluate_ndcg_at_10(run_data.qrels, fused_rankings, cutoff=10)
fused_evaluation_seconds = time.perf_counter() - evaluation_started
print(json.dumps({
    'e5_ndcg_at_10': baseline_metric['ndcg_at_10'],
    'e5_rerank_raw_ndcg_at_10': raw_reranked_metric['ndcg_at_10'],
    'e5_rerank_fused_ndcg_at_10': reranked_metric['ndcg_at_10'],
    'delta_vs_e5': reranked_metric['ndcg_at_10'] - baseline_metric['ndcg_at_10'],
    'fusion_weights': list(config.fusion_weights),
}, indent=2))

## 8. Persist comparison and provenance

The result reports the validation-selected E5/reranker fusion as the reranking
system score and preserves the raw cross-encoder score as a diagnostic. Smoke
results are not benchmark evidence.

In [ ]:
environment = environment_metadata(config, repo_root=workspace_root)
result = build_comparison_result(
    config,
    data,
    run_data,
    baseline_metric,
    reranked_metric,
    raw_reranked_metric=raw_reranked_metric,
    identity=identity,
    environment=environment,
    timings={
        'corpus_encoding': corpus_seconds,
        'query_encoding': query_seconds,
        'first_stage_ranking': ranking_seconds,
        'cross_encoder_reranking': reranking_seconds,
        'baseline_evaluation': baseline_evaluation_seconds,
        'raw_reranked_evaluation': raw_reranked_evaluation_seconds,
        'fused_evaluation': fused_evaluation_seconds,
        'rrf_fusion': fusion_seconds,
    },
    candidate_pool_was_preserved=pool_preserved,
    notebook_sha256=notebook_sha256,
    cache_paths={name: str(path) for name, path in paths.items()},
    repo_root=workspace_root,
)
artifact_paths = write_result_artifacts(
    config,
    result,
    metadata={
        'result': result,
        'cache_metadata': {
            'corpus': corpus_metadata,
            'queries': query_metadata,
            'rankings': ranking_metadata,
            'reranked_rankings': rerank_metadata,
            'fused_rankings': fused_metadata,
        },
    },
)
print(json.dumps({
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'e5_nDCG@10': result['systems']['e5']['ndcg_at_10'],
    'e5_rerank_nDCG@10': result['systems']['e5_rerank']['ndcg_at_10'],
    'delta_nDCG@10': result['delta_ndcg_at_10'],
    'candidate_pool_preserved': result['candidate_pool']['preserved'],
    'component_diagnostics': result['component_diagnostics'],
    'fusion': result['fusion'],
    'artifact_paths': artifact_paths,
}, indent=2))

## Interpretation boundary

Use only a result with `status: benchmark` and `benchmark_evidence: true` for the
primary E5 versus E5 + reranking comparison. The comparison uses a valid-selected
fusion weight, preserves the full E5 depth-1,000 candidate set, and retains raw
cross-encoder output as a diagnostic. Smoke scores are wiring evidence only.